# Prior-knowledge-source ablation

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
    "#000000",
    "#ffcc00",
    "#8b4513",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
    "schwarz",
    "goldgelb",
    "sattelbraun",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig2b_neurips2021_cite_bmmc_scib_ablations")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from adjustText import adjust_text
import matplotlib.patches as mpatches
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


# Ablation

In [ ]:
EXPERIMENTS = "../.experiment_data/primary/experiments_new/"
RERUNS = "../.experiment_data/reruns/fig2b/"
PREFIX = "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_"
batch_key = "Site"

reference_paths = [EXPERIMENTS + PREFIX + "ENCFF173VDJ_gaf_standard_with_rna_s0/"] + [
    RERUNS + f"mmd_default_r{run}/experiments_new/" + PREFIX + "ENCFF173VDJ_gaf_standard_with_rna_s0/" for run in [1, 2, 3]
]

tools = {
    "Pathway Commons": "ENCFF173VDJ_gaf_pcommons_filtont_pathway_s0",
    "Reactome": "ENCFF173VDJ_gaf_reactome_filtont_reactome_s0",
    "GRAND GRN (K562)": "grand_k562_grn_gaf_standard_with_rna_s0",
    "BioGRID": "biogrid_gaf_standard_with_rna_s0",
    "STRING": "string_gaf_standard_with_rna_s0",
    "GM12878 (ENCFF203AKP)": "ENCFF203AKP_gaf_standard_with_rna_s0",
    "T47D (ENCFF804SET)": "ENCFF804SET_gaf_standard_with_rna_s0",
    "MCF-7 (ENCFF164AGX)": "ENCFF164AGX_gaf_standard_with_rna_s0",
    "K562 Position-shifted TAD benchmark": "ENCFF173VDJ_shifted_benchmark_gaf_standard_with_rna_s0",
    "K562 interactions shuffle": "ENCFF173VDJ_shuff_gaf_standard_with_rna_s0",
    "TFLink": "tflink_gaf_standard_with_rna_s0",
    "hTFtarget": "htftarget_bone_marrow_gaf_standard_with_rna_s0",
    "GO BP 2014": "ENCFF173VDJ_gaf_go2014_s0",
    "GO BP 2019": "ENCFF173VDJ_gaf_go2019_s0",
    "GO BP 2022": "ENCFF173VDJ_gaf_go2022_s0",
    "Gene-GO ass. shuffle": "ENCFF173VDJ_gaf_standard_with_rna_gshuff_s0",
    "GO CC": "ENCFF173VDJ_gaf_standard_with_rna_filtont_cellular_component_s0",
    "GO MF": "ENCFF173VDJ_gaf_standard_with_rna_filtont_molecular_function_s0",
}


def get_scib(path):
    scib_label = float(pd.read_csv(os.path.join(path, "default", "scib_results_label.csv"))["Bio conservation"].iloc[0])
    scib_batch = float(pd.read_csv(os.path.join(path, "default", f"scib_results_{batch_key}.csv"))["Batch correction"].iloc[0])
    return 0.6 * scib_label + 0.4 * scib_batch


tool_dirs = {label: EXPERIMENTS + PREFIX + name + "/" for label, name in tools.items()}
tool_dirs["hTFtarget"] = RERUNS + "mmd_alt_htftarget_r1/experiments_new/" + PREFIX + tools["hTFtarget"] + "/"
tool_dirs["GO MF"] = RERUNS + "mmd_alt_go_mf_r1/experiments_new/" + PREFIX + tools["GO MF"] + "/"

reference_scib = np.array([get_scib(path) for path in reference_paths])
results = pd.DataFrame(
    {
        "tool_label": ["NetworkVI"] * len(reference_paths) + list(tools),
        "scib": list(reference_scib) + [get_scib(path) for path in tool_dirs.values()],
    }
)
results["delta_scib"] = results["scib"] - reference_scib.mean()
print("NetworkVI default runs:", np.round(reference_scib, 4), f"mean = {reference_scib.mean():.4f}, SD = {reference_scib.std(ddof=1):.4f}")
results.to_csv(os.path.join(_OUTDIR_ABS, "fig2b_scib_mmd_reference_mean.csv"), index=False)
results.round(4)

In [ ]:
BASELINE_LABEL = 'NetworkVI'
networkvi_scib = results.loc[results['tool_label'] == BASELINE_LABEL, 'scib'].mean()
SECTIONS = [('GO versions', ['GO BP 2014', 'GO BP 2019', 'GO BP 2022'], 'go'), ('GO\nname-\nspaces', ['GO MF', 'GO CC'], 'go'), ('Pathway\nontologies', ['Pathway Commons', 'Reactome'], 'pathway'), ('Non-specific\nnetworks', ['STRING', 'BioGRID', 'GRAND GRN (K562)', 'TFLink', 'hTFtarget'], 'tad'), ('Alt. cell-line\nTADs', ['GM12878 (ENCFF203AKP)', 'T47D (ENCFF804SET)', 'MCF-7 (ENCFF164AGX)'], 'tad'), ('Negative\nControl', ['K562 Position-shifted TAD benchmark', 'K562 interactions shuffle', 'Gene-GO ass. shuffle'], 'shuffle')]
TICK_RENAME = {'GM12878 (ENCFF203AKP)': 'GM12878 (lymphoid)', 'T47D (ENCFF804SET)': 'T47D (breast)', 'MCF-7 (ENCFF164AGX)': 'MCF-7 (breast)', 'K562 Position-shifted TAD benchmark': 'K562 Position-shifted'}
PALETTE = {'go': '#A0522D', 'pathway': '#5F8575', 'tad': '#6A5ACD', 'shuffle': '#D2691E'}
scib_lookup = results.set_index('tool_label')['scib'].to_dict()
bar_labels = []
bar_deltas = []
bar_colours = []
section_spans = {}
x = 0
for s_idx, (sec_name, tools, group) in enumerate(SECTIONS):
    x_start = x
    for tool in tools:
        scib_val = scib_lookup[tool]
        bar_labels.append(tool)
        bar_deltas.append(scib_val - networkvi_scib)
        bar_colours.append(PALETTE[group])
        x += 1
    x_end = x - 1
    section_spans[sec_name] = (x_start, x_end)
n_bars = len(bar_labels)
xs = np.arange(n_bars)
fig_width = 1 / 2 * 7.0
fig_height = 2
set_figure_params(fig_width, fig_height)
fig, ax = plt.subplots()
Y_CLIP_MIN = -0.2
bar_deltas_clipped = [max(d, Y_CLIP_MIN) for d in bar_deltas]
needs_break = [d < Y_CLIP_MIN for d in bar_deltas]
bar_width = 0.6
bars = ax.bar(xs, bar_deltas_clipped, color=bar_colours, width=bar_width, edgecolor='white', linewidth=0.3, zorder=4)
for xi, (clipped, broken, delta) in enumerate(zip(bar_deltas_clipped, needs_break, bar_deltas)):
    if broken:
        slash_y = Y_CLIP_MIN + 0.003
        for offset in (-0.08, 0.08):
            ax.plot([xi + offset - 0.06, xi + offset + 0.06], [slash_y - 0.008, slash_y + 0.008], color='white', linewidth=1.2, zorder=6)
            ax.plot([xi + offset - 0.06, xi + offset + 0.06], [slash_y - 0.016, slash_y], color=bar_colours[xi], linewidth=1.2, zorder=6)
ax.axhline(0, color='black', linewidth=0.8, zorder=3)
trans = ax.get_xaxis_transform()
BRACKET_Y = 1.06
TICK_DROP = 0.03
LABEL_Y = 1.1
SECTION_DISPLAY = {'GO versions': 'GO versions', 'GO\nnamespaces': 'GO\nnamespaces', 'Pathway\nontologies': 'Pathway\nontologies', 'Non-specific\nnetworks': 'Non-specific\nnetworks', 'Alt. cell-line\nTADs': 'Alt. cell-line\nTADs', 'Shuffle\ncontrol': 'Control'}

def bracket_half(x_start, x_end):
    n = x_end - x_start + 1
    return 0.2 if n <= 2 else 0.28
for sec_name, (x_start, x_end) in section_spans.items():
    bh = bracket_half(x_start, x_end)
    lx = x_start - bh
    rx = x_end + bh
    mid = (lx + rx) / 2.0
    label = SECTION_DISPLAY.get(sec_name, sec_name)
    ax.plot([lx, rx], [BRACKET_Y, BRACKET_Y], transform=trans, color='#555555', linewidth=0.7, clip_on=False, zorder=10)
    ax.plot([lx, lx], [BRACKET_Y - TICK_DROP, BRACKET_Y], transform=trans, color='#555555', linewidth=0.7, clip_on=False, zorder=10)
    ax.plot([rx, rx], [BRACKET_Y - TICK_DROP, BRACKET_Y], transform=trans, color='#555555', linewidth=0.7, clip_on=False, zorder=10)
    ax.text(mid, LABEL_Y, label, transform=trans, ha='center', va='bottom', fontsize=mpl.rcParams['font.size'] - 0.5, color='#333333', multialignment='center', linespacing=1.2, clip_on=False)
tick_labels = [TICK_RENAME.get(l, l) for l in bar_labels]
ax.set_xticks(xs)
ax.set_xticklabels(tick_labels, rotation=40, ha='right')
ax.set_ylabel('ΔscIB score vs. NetworkVI')
delta_min_vis = min(bar_deltas_clipped)
delta_max_vis = 0.05
rng = delta_max_vis - delta_min_vis
ax.set_ylim(delta_min_vis - rng * 0.18, delta_max_vis)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(-0.75, n_bars - 0.25)
out_pdf = 'networkvi_ablation_prior_knowledge_mmd_reference_mean.pdf'
out_png = 'networkvi_ablation_prior_knowledge_mmd_reference_mean.png'
out_svg = 'networkvi_ablation_prior_knowledge_mmd_reference_mean.svg'
fig.savefig(os.path.join(_OUTDIR_ABS, out_pdf), dpi=500, bbox_inches='tight', transparent=True)
fig.savefig(os.path.join(_OUTDIR_ABS, out_png), dpi=500, bbox_inches='tight', transparent=True)
fig.savefig(os.path.join(_OUTDIR_ABS, out_svg), format='svg', bbox_inches='tight', transparent=True)
print(f'Saved:\n  {out_pdf}\n  {out_png}\n  {out_svg}')


In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def analyze_graph_pkls(base_dir, threshold=0.55, verbose=True):
    '''Find all *_graph_csv_with_ci.pkl files under base_dir,'''

    results = []

    for root, _, files in os.walk(base_dir):
        for f in tqdm(files):
            if f.endswith("_graph_csv_with_ci.pkl"):
                path = os.path.join(root, f)

                try:
                    df = pd.read_pickle(path)

                    total_terms = len(df)
                    if total_terms == 0:
                        continue

                    frac_above = (df["predictors_label_valid_roc"] > threshold).sum() / total_terms

                    results.append(
                        {"file": path, "n_terms": total_terms, "frac_above_threshold": frac_above}
                    )

                    if verbose:
                        print(f"\n{path}")
                        print(f"  Terms: {total_terms}")
                        print(f"  Fraction > {threshold}: {frac_above:.4f}")

                except Exception as e:
                    print(f"Failed to load {path}: {e}")

    if not results:
        print("No matching files found.")
        return [], None, None

    mean_fraction = np.mean([r["frac_above_threshold"] for r in results])
    mean_terms = np.mean([r["n_terms"] for r in results])

    print("\n=== SUMMARY ===")
    print(f"Files analyzed: {len(results)}")
    print(f"Mean #terms: {mean_terms:.2f}")
    print(f"Mean fraction > {threshold}: {mean_fraction:.4f}")

    return results, mean_fraction, mean_terms


In [ ]:
base_path = "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_pcommons_filtont_pathway_s0/default/calc_go_terms_cell_type/"
results, mean_frac, mean_terms = analyze_graph_pkls(base_path)


In [ ]:
base_path = "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_reactome_filtont_reactome_s0/"
results, mean_frac, mean_terms = analyze_graph_pkls(base_path)


In [ ]:
base_path = "../.experiment_data/reruns/fig2b/mmd_alt_go_mf_r1/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_filtont_molecular_function_s0/default/calc_go_terms_cell_type/"
results, mean_frac, mean_terms = analyze_graph_pkls(base_path)

In [ ]:
base_path = "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_filtont_cellular_component_s0/"
results, mean_frac, mean_terms = analyze_graph_pkls(base_path)


# Stability

In [ ]:
graph_csv = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_subsam_0.5_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_B1 B IGKC+_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)
graph_csv


In [ ]:
graph_csv1 = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_B1 B IGKC+_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)
graph_csv1


In [ ]:
go_one_over055 = list(
    set(graph_csv[graph_csv["predictors_label_valid_roc"] > 0.65].index)
    & set(graph_csv1[graph_csv1["predictors_label_valid_roc"] > 0.65].index)
)


In [ ]:
scipy.stats.spearmanr(
    graph_csv["predictors_label_valid_roc"], graph_csv1["predictors_label_valid_roc"]
)


In [ ]:
scipy.stats.spearmanr(
    graph_csv["predictors_label_valid_roc"].loc[go_one_over055],
    graph_csv1["predictors_label_valid_roc"].loc[go_one_over055],
)


In [ ]:
"""Full stability analysis: all pairwise comparisons across seeds and subsampling."""
import os, re, glob
import numpy as np
import pandas as pd
from itertools import combinations
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr
_BASE = '../.experiment_data/scratch/experiments_new/'
_SEED = 'sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_{s}/default/calc_go_terms_cell_type/'
_SUB = 'sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_subsam_{f}_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/'
MODELS = [('Seed 0\n(base)', _BASE + _SEED.format(s='s0'), 'seed'), ('Seed 1', _BASE + _SEED.format(s='s1'), 'seed'), ('Seed 2', _BASE + _SEED.format(s='s2'), 'seed'), ('50 % cells', _BASE + _SUB.format(f='0.5'), 'subsample'), ('75 % cells', _BASE + _SUB.format(f='0.75'), 'subsample')]
CELL_TYPES_FILTER = ['CD8+ T CD49f+', 'CD8+ T TIGIT+ CD45RO+', 'Erythroblast', 'Normoblast', 'CD14+ Mono', 'CD16+ Mono', 'Naive CD20+ B IGKC+', 'NK', 'pDC', 'CD4+ T naive', 'CD8+ T naive']
GROUP_PALETTE = {'seed': '#8B6BAE', 'subsample': '#5F8575'}
AUROC_COL = 'predictors_label_valid_roc'

def _canonical(name: str) -> str:
    """'G/M prog' == 'G_M prog' == 'G_M_prog'  →  'g m prog'"""
    s = name.lower()
    s = re.sub('[/_]', ' ', s)
    s = re.sub(' +', ' ', s).strip()
    return s
_PKL_RE = re.compile('^(expression|protein)_cell_type_(vs_rest|vs_other_t_cells)_(.+?)_res_unp_act_mask_False_flip_False_go_terms(?:_genes)?_evaluated_graph_csv_with_ci\\.pkl$')

def _parse(fname):
    m = _PKL_RE.match(os.path.basename(fname))
    return (m.group(1), m.group(2), m.group(3)) if m else None

def load_model(calc_dir, filter_phenotypes=None, modality='expression', comparison_type='vs_rest', verbose=True):
    """Load all matching pkls → {canonical_key: DataFrame}."""
    print(calc_dir)
    pkls = sorted(glob.glob(os.path.join(calc_dir, '*.pkl')))
    if not pkls:
        raise FileNotFoundError(f'No pkls in: {calc_dir}')
    filter_canon = {_canonical(p) for p in filter_phenotypes} if filter_phenotypes else None
    out = {}
    for path in pkls:
        parsed = _parse(path)
        if not parsed:
            continue
        mod, ctype, raw = parsed
        if modality is not None and mod != modality:
            continue
        if comparison_type is not None and ctype != comparison_type:
            continue
        canon = _canonical(raw)
        if filter_canon is not None and canon not in filter_canon:
            continue
        try:
            df = pd.read_pickle(path)
        except Exception as e:
            if verbose:
                print(f'  ⚠ {os.path.basename(path)}: {e}')
            continue
        for col in ['go_term', 'GO_term', 'go_id', 'term']:
            if col in df.columns:
                df = df.set_index(col)
                break
        out[canon] = df
    if verbose:
        label = os.path.basename(calc_dir.rstrip('/'))[-30:]
        print(f'  {len(out):2d} phenotypes  ←  ...{label}')
    return out

def spearman_stats(x, y):
    x, y = (np.asarray(x, float), np.asarray(y, float))
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = (x[ok], y[ok])
    if len(x) < 5:
        return dict(rho=np.nan, p_value=np.nan, n=len(x))
    rho, p = spearmanr(x, y)
    return dict(rho=float(rho), p_value=float(p), n=len(x))

def compare_pair(data_a, data_b, min_go=10):
    """Compute per-phenotype Spearman ρ for one model pair."""
    shared = sorted(set(data_a) & set(data_b))
    rows = []
    for ph in shared:
        shared_go = data_a[ph].index.intersection(data_b[ph].index)
        if len(shared_go) < min_go:
            continue
        if AUROC_COL not in data_a[ph].columns:
            continue
        x = data_a[ph].loc[shared_go, AUROC_COL].values.astype(float)
        y = data_b[ph].loc[shared_go, AUROC_COL].values.astype(float)
        s = spearman_stats(x, y)
        s['phenotype'] = ph
        s['n_go'] = len(shared_go)
        rows.append(s)
    return pd.DataFrame(rows).set_index('phenotype') if rows else pd.DataFrame()

def _stars(p):
    if np.isnan(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'

def load_all_models(models, filter_phenotypes, verbose=True):
    """Returns list of (label, group, data_dict)."""
    loaded = []
    for label, path, group in models:
        if verbose:
            print(f'Loading {label!r} …')
        try:
            data = load_model(path, filter_phenotypes=filter_phenotypes, verbose=verbose)
            loaded.append((label, group, data))
        except FileNotFoundError as e:
            print(f'  ⚠ skipped: {e}')
    return loaded

def pairwise_summary(loaded_models, display_labels=None):
    '''Tidy DataFrame: one row per (model_A, model_B, phenotype).'''
    rows = []
    for (la, ga, da), (lb, gb, db) in combinations(loaded_models, 2):
        df = compare_pair(da, db)
        if df.empty:
            continue
        df = df.copy()
        df['model_a'] = la.replace('\n', ' ')
        df['model_b'] = lb.replace('\n', ' ')
        df['stars'] = df['p_value'].apply(_stars)
        if display_labels:
            df.index = [display_labels.get(p, p) for p in df.index]
        rows.append(df.reset_index().rename(columns={'index': 'phenotype'}))
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    cols = ['model_a', 'model_b', 'phenotype', 'rho', 'p_value', 'stars', 'n', 'n_go']
    return out[[c for c in cols if c in out.columns]]
if __name__ == '__main__':
    os.makedirs(_OUTDIR_ABS, exist_ok=True)
    print('Loading models …')
    loaded = load_all_models(MODELS, filter_phenotypes=CELL_TYPES_FILTER)
    display_labels = {_canonical(p): p for p in CELL_TYPES_FILTER}
    tbl = pairwise_summary(loaded, display_labels=display_labels)
    print(tbl.to_string(index=False))
    print('Done.')
